<a href="https://colab.research.google.com/github/elhartw/ML4CHEM/blob/main/chemberta_bbbp_improved.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 1. GPU check

In [1]:
import torch
print("CUDA beschikbaar:", torch.cuda.is_available())
print("GPU naam:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "geen GPU")

CUDA beschikbaar: True
GPU naam: Tesla T4


## 2. Installaties

In [6]:
!pip install -q transformers rdkit scikit-learn pandas numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.0/37.0 MB 20.0 MB/s eta 0:00:00


## 3. Data laden

In [7]:
import pandas as pd

URL = "https://deepchemdata.s3-us-west-1.amazonaws.com/datasets/BBBP.csv"
df = pd.read_csv(URL)

print("Aantal rijen:", len(df))
print("Kolommen:", df.columns.tolist())
df.head()

Aantal rijen: 2050
Kolommen: ['num', 'name', 'p_np', 'smiles']


,num,name,p_np,smiles
0,1,Propanolol,1,[Cl].CC(C)NCC(O)COc1cccc2ccccc12
1,2,Terbutylchlorambucil,1,C(=O)(OC(C)(C)C)CCCc1ccc(cc1)N(CCCl)CCCl
2,3,40730,1,c12c3c(N4CCN(C)CC4)c(F)cc1c(c(C(O)=O)cn2C(C)CO...
3,4,24,1,C1CCN(CC1)Cc1cccc(c1)OCCCNC(=O)C
4,5,cloxacillin,1,Cc1onc(c2ccccc2Cl)c1C(=O)N[C@H]3[C@H]4SC(C)(C)...


In [8]:
print(df["p_np"].value_counts())
print("\nFractie positief:", df["p_np"].mean().round(3))

p_np
1    1567
0     483
Name: count, dtype: int64

Fractie positief: 0.764


## 4. SMILES opschonen: kanonieke SMILES via RDKit




In [9]:
from rdkit import Chem

def canonicalize(smiles):
    """Zet SMILES om naar kanonieke vorm. Geeft None terug bij ongeldige SMILES."""
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    return Chem.MolToSmiles(mol)

df["smiles"] = df["smiles"].apply(canonicalize)

# Verwijder rijen met ongeldige SMILES
n_before = len(df)
df = df.dropna(subset=["smiles"]).reset_index(drop=True)
print(f"Verwijderd wegens ongeldige SMILES: {n_before - len(df)}")
print(f"Overgebleven: {len(df)} moleculen")

[17:08:14] Explicit valence for atom # 1 N, 4, is greater than permitted
[17:08:14] WARNING: not removing hydrogen atom without neighbors
[17:08:14] Explicit valence for atom # 6 N, 4, is greater than permitted
[17:08:14] WARNING: not removing hydrogen atom without neighbors
[17:08:14] WARNING: not removing hydrogen atom without neighbors
[17:08:14] WARNING: not removing hydrogen atom without neighbors
[17:08:14] WARNING: not removing hydrogen atom without neighbors
[17:08:14] WARNING: not removing hydrogen atom without neighbors
[17:08:14] WARNING: not removing hydrogen atom without neighbors
[17:08:14] Explicit valence for atom # 6 N, 4, is greater than permitted
[17:08:14] WARNING: not removing hydrogen atom without neighbors
[17:08:14] WARNING: not removing hydrogen atom without neighbors
[17:08:14] WARNING: not removing hydrogen atom without neighbors
[17:08:14] WARNING: not removing hydrogen atom without neighbors
[17:08:14] Explicit valence for atom # 11 N, 4, is greater than pe

Verwijderd wegens ongeldige SMILES: 11
Overgebleven: 2039 moleculen


[17:08:14] WARNING: not removing hydrogen atom without neighbors
[17:08:14] WARNING: not removing hydrogen atom without neighbors
[17:08:14] WARNING: not removing hydrogen atom without neighbors
[17:08:14] WARNING: not removing hydrogen atom without neighbors
[17:08:14] WARNING: not removing hydrogen atom without neighbors
[17:08:14] WARNING: not removing hydrogen atom without neighbors
[17:08:14] WARNING: not removing hydrogen atom without neighbors
[17:08:14] WARNING: not removing hydrogen atom without neighbors
[17:08:14] WARNING: not removing hydrogen atom without neighbors
[17:08:14] WARNING: not removing hydrogen atom without neighbors


## 5. Scaffold split



In [10]:
from rdkit.Chem.Scaffolds import MurckoScaffold
from collections import defaultdict
import numpy as np

def get_scaffold(smiles):
    """Geeft de Murcko scaffold van een SMILES string."""
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return smiles  # fallback
    scaffold = MurckoScaffold.GetScaffoldForMol(mol)
    return Chem.MolToSmiles(scaffold)


def scaffold_split(df, frac_train=0.8, frac_val=0.1, frac_test=0.1, seed=42):
    """
    Splitst DataFrame op basis van Murcko scaffolds.
    Moleculen met dezelfde scaffold blijven altijd in dezelfde set.
    """
    assert abs(frac_train + frac_val + frac_test - 1.0) < 1e-6

    # Bereken scaffold per molecuul
    scaffolds = defaultdict(list)
    for idx, smi in enumerate(df["smiles"]):
        scaffold = get_scaffold(smi)
        scaffolds[scaffold].append(idx)

    # Sorteer scaffolds op grootte: groot → klein
    scaffold_sets = sorted(scaffolds.values(), key=len, reverse=True)

    n = len(df)
    train_cutoff = frac_train * n
    val_cutoff   = (frac_train + frac_val) * n

    train_idx, val_idx, test_idx = [], [], []

    for group in scaffold_sets:
        if len(train_idx) < train_cutoff:
            train_idx.extend(group)
        elif len(train_idx) + len(val_idx) < val_cutoff:
            val_idx.extend(group)
        else:
            test_idx.extend(group)

    rng = np.random.default_rng(seed)
    rng.shuffle(train_idx)
    rng.shuffle(val_idx)
    rng.shuffle(test_idx)

    return (
        df.iloc[train_idx].reset_index(drop=True),
        df.iloc[val_idx].reset_index(drop=True),
        df.iloc[test_idx].reset_index(drop=True),
    )

In [11]:
train_df, val_df, test_df = scaffold_split(df)

print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")
print(f"\nKlassebalans train: {train_df['p_np'].mean():.3f} positief")
print(f"Klassebalans val:   {val_df['p_np'].mean():.3f} positief")
print(f"Klassebalans test:  {test_df['p_np'].mean():.3f} positief")

# Controleer scaffold-overlap
train_scaffolds = set(get_scaffold(s) for s in train_df["smiles"])
test_scaffolds  = set(get_scaffold(s) for s in test_df["smiles"])
overlap = train_scaffolds & test_scaffolds
print(f"\nScaffold overlap train/test: {len(overlap)} (moet 0 zijn)")

[17:08:31] WARNING: not removing hydrogen atom without neighbors
[17:08:31] WARNING: not removing hydrogen atom without neighbors
[17:08:31] WARNING: not removing hydrogen atom without neighbors
[17:08:31] WARNING: not removing hydrogen atom without neighbors
[17:08:31] WARNING: not removing hydrogen atom without neighbors
[17:08:31] WARNING: not removing hydrogen atom without neighbors
[17:08:31] WARNING: not removing hydrogen atom without neighbors
[17:08:31] WARNING: not removing hydrogen atom without neighbors
[17:08:31] WARNING: not removing hydrogen atom without neighbors
[17:08:31] WARNING: not removing hydrogen atom without neighbors
[17:08:31] WARNING: not removing hydrogen atom without neighbors
[17:08:31] WARNING: not removing hydrogen atom without neighbors
[17:08:31] WARNING: not removing hydrogen atom without neighbors
[17:08:31] WARNING: not removing hydrogen atom without neighbors
[17:08:31] WARNING: not removing hydrogen atom without neighbors
[17:08:31] WARNING: not r

Train: 1632 | Val: 204 | Test: 203

Klassebalans train: 0.706 positief
Klassebalans val:   1.000 positief
Klassebalans test:  1.000 positief


[17:08:32] WARNING: not removing hydrogen atom without neighbors
[17:08:33] WARNING: not removing hydrogen atom without neighbors
[17:08:33] WARNING: not removing hydrogen atom without neighbors
[17:08:33] WARNING: not removing hydrogen atom without neighbors
[17:08:33] WARNING: not removing hydrogen atom without neighbors
[17:08:33] WARNING: not removing hydrogen atom without neighbors
[17:08:33] WARNING: not removing hydrogen atom without neighbors
[17:08:33] WARNING: not removing hydrogen atom without neighbors
[17:08:33] WARNING: not removing hydrogen atom without neighbors
[17:08:33] WARNING: not removing hydrogen atom without neighbors
[17:08:33] WARNING: not removing hydrogen atom without neighbors
[17:08:33] WARNING: not removing hydrogen atom without neighbors
[17:08:33] WARNING: not removing hydrogen atom without neighbors
[17:08:33] WARNING: not removing hydrogen atom without neighbors
[17:08:33] WARNING: not removing hydrogen atom without neighbors
[17:08:33] WARNING: not r


Scaffold overlap train/test: 0 (moet 0 zijn)


[17:08:33] WARNING: not removing hydrogen atom without neighbors
[17:08:33] WARNING: not removing hydrogen atom without neighbors
[17:08:33] WARNING: not removing hydrogen atom without neighbors
[17:08:33] WARNING: not removing hydrogen atom without neighbors
[17:08:33] WARNING: not removing hydrogen atom without neighbors
[17:08:33] WARNING: not removing hydrogen atom without neighbors
[17:08:33] WARNING: not removing hydrogen atom without neighbors


In [12]:
train_smiles = train_df["smiles"].tolist()
train_labels = train_df["p_np"].astype(int).tolist()

val_smiles   = val_df["smiles"].tolist()
val_labels   = val_df["p_np"].astype(int).tolist()

test_smiles  = test_df["smiles"].tolist()
test_labels  = test_df["p_np"].astype(int).tolist()

## 6. Model laden: ChemBERTa-77M-MTR


In [13]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, RobertaConfig

MODEL_NAME = "DeepChem/ChemBERTa-77M-MTR"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Hogere dropout op de classifier head
config = RobertaConfig.from_pretrained(MODEL_NAME, num_labels=2, classifier_dropout=0.3)
model  = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, config=config)

print("Model geladen:", MODEL_NAME)
print("Classifier dropout:", config.classifier_dropout)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/420 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/14.0M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/53 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: DeepChem/ChemBERTa-77M-MTR
Key                             | Status     | 
--------------------------------+------------+-
roberta.embeddings.position_ids | UNEXPECTED | 
regression.dense.bias           | UNEXPECTED | 
norm_std                        | UNEXPECTED | 
regression.out_proj.weight      | UNEXPECTED | 
regression.out_proj.bias        | UNEXPECTED | 
norm_mean                       | UNEXPECTED | 
regression.dense.weight         | UNEXPECTED | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.bias           | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model geladen: DeepChem/ChemBERTa-77M-MTR
Classifier dropout: 0.3


## 7. SMILES augmentatie



In [14]:
import random

def randomize_smiles(smiles, n_tries=5):
    """
    Genereert een willekeurige SMILES voor dezelfde molecule.
    Geeft de originele terug als randomisatie mislukt.
    """
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return smiles
    for _ in range(n_tries):
        # Willekeurig startpunt kiezen
        atom_order = list(range(mol.GetNumAtoms()))
        random.shuffle(atom_order)
        new_mol = Chem.RenumberAtoms(mol, atom_order)
        new_smi = Chem.MolToSmiles(new_mol, canonical=False)
        if new_smi and new_smi != smiles:
            return new_smi
    return smiles

# Voorbeeld
aspirin = "CC(=O)Oc1ccccc1C(=O)O"
print("Origineel:", aspirin)
for i in range(3):
    print(f"Augmentatie {i+1}:", randomize_smiles(aspirin))

Origineel: CC(=O)Oc1ccccc1C(=O)O
Augmentatie 1: CC(Oc1ccccc1C(=O)O)=O
Augmentatie 2: c1ccc(C(O)=O)c(OC(=O)C)c1
Augmentatie 3: O(c1c(C(O)=O)cccc1)C(=O)C


## 8. Dataset klasse met augmentatie

In [15]:
import torch
from torch.utils.data import Dataset

class SMILESDataset(Dataset):
    def __init__(self, smiles, labels, tokenizer, max_length=128, augment=False):
        self.smiles     = smiles
        self.labels     = labels
        self.tokenizer  = tokenizer
        self.max_length = max_length
        self.augment    = augment  # alleen True voor trainset

    def __len__(self):
        return len(self.smiles)

    def __getitem__(self, idx):
        smi = self.smiles[idx]

        # Augmentatie: willekeurig andere SMILES voor dezelfde molecule
        if self.augment and random.random() < 0.5:
            smi = randomize_smiles(smi)

        enc = self.tokenizer(
            smi,
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt",
        )
        return {
            "input_ids":      enc["input_ids"].flatten(),
            "attention_mask": enc["attention_mask"].flatten(),
            "labels":         torch.tensor(self.labels[idx], dtype=torch.long),
        }

# augment=True alleen voor train
train_ds = SMILESDataset(train_smiles, train_labels, tokenizer, augment=True)
val_ds   = SMILESDataset(val_smiles,   val_labels,   tokenizer, augment=False)
test_ds  = SMILESDataset(test_smiles,  test_labels,  tokenizer, augment=False)

print(f"Train: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}")

Train: 1632 | Val: 204 | Test: 203


## 9. Klasse gewichten voor klasse-onbalans

In [16]:
from sklearn.utils.class_weight import compute_class_weight

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.array([0, 1]),
    y=train_labels,
)
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float)
print("Klasse gewichten:", class_weights_tensor)

Klasse gewichten: tensor([1.7035, 0.7077])


## 10. Custom Trainer met gewogen loss + label smoothing

In [17]:
import torch.nn as nn
from transformers import Trainer

class WeightedTrainer(Trainer):
    """
    Trainer met:
    - Gewogen cross-entropy voor klasse-onbalans
    - Label smoothing tegen overfitten
    """
    def __init__(self, class_weights, label_smoothing=0.1, **kwargs):
        super().__init__(**kwargs)
        self.class_weights    = class_weights
        self.label_smoothing  = label_smoothing

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits  = outputs.logits

        # Gewogen cross-entropy met label smoothing
        loss_fn = nn.CrossEntropyLoss(
            weight=self.class_weights.to(logits.device),
            label_smoothing=self.label_smoothing,
        )
        loss = loss_fn(logits, labels)

        return (loss, outputs) if return_outputs else loss

## 11. Metrics: accuracy, ROC-AUC


In [23]:
import numpy as np
from sklearn.metrics import accuracy_score, roc_auc_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    probs = torch.softmax(torch.tensor(logits), dim=-1)[:, 1].numpy()
    return {
        "accuracy": accuracy_score(labels, preds),
        "roc_auc":  roc_auc_score(labels, probs),}

## 12. Training

In [19]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./chemberta_bbbp_v2",
    num_train_epochs=10,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_steps=50,
    lr_scheduler_type="cosine",        # cosine i.p.v. lineair
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="roc_auc",
    greater_is_better=True,
    logging_steps=20,
    save_total_limit=2,
    report_to="none",
    fp16=True,
)

trainer = WeightedTrainer(
    class_weights=class_weights_tensor,
    label_smoothing=0.1,
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
)

In [20]:
trainer.train()

[17:11:56] WARNING: not removing hydrogen atom without neighbors


Epoch,Training Loss,Validation Loss,Accuracy,Roc Auc,Mcc
1,0.707628,0.730046,0.823529,nan,0.000000
2,0.683567,0.718636,0.828431,nan,0.000000
3,0.656894,0.704036,0.794118,nan,0.000000
4,0.614197,0.679861,0.799020,nan,0.000000
5,0.578605,0.644978,0.862745,nan,0.000000
6,0.553559,0.620756,0.872549,nan,0.000000
7,0.554451,0.600246,0.887255,nan,0.000000
8,0.535634,0.592062,0.892157,nan,0.000000
9,0.529442,0.588269,0.892157,nan,0.000000
10,0.532901,0.587974,0.892157,nan,0.000000


[17:11:58] WARNING: not removing hydrogen atom without neighbors
[17:11:58] WARNING: not removing hydrogen atom without neighbors
[17:11:58] WARNING: not removing hydrogen atom without neighbors
[17:11:58] WARNING: not removing hydrogen atom without neighbors
[17:11:58] WARNING: not removing hydrogen atom without neighbors
[17:11:58] WARNING: not removing hydrogen atom without neighbors
[17:11:58] WARNING: not removing hydrogen atom without neighbors
[17:11:58] WARNING: not removing hydrogen atom without neighbors
[17:11:59] WARNING: not removing hydrogen atom without neighbors
[17:11:59] WARNING: not removing hydrogen atom without neighbors
[17:11:59] WARNING: not removing hydrogen atom without neighbors
[17:11:59] WARNING: not removing hydrogen atom without neighbors
[17:11:59] WARNING: not removing hydrogen atom without neighbors
[17:11:59] WARNING: not removing hydrogen atom without neighbors
[17:11:59] WARNING: not removing hydrogen atom without neighbors
/usr/local/lib/python3.12

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[17:12:00] WARNING: not removing hydrogen atom without neighbors
[17:12:00] WARNING: not removing hydrogen atom without neighbors
[17:12:00] WARNING: not removing hydrogen atom without neighbors
[17:12:01] WARNING: not removing hydrogen atom without neighbors
[17:12:01] WARNING: not removing hydrogen atom without neighbors
[17:12:01] WARNING: not removing hydrogen atom without neighbors
[17:12:01] WARNING: not removing hydrogen atom without neighbors
[17:12:01] WARNING: not removing hydrogen atom without neighbors
[17:12:01] WARNING: not removing hydrogen atom without neighbors
[17:12:01] WARNING: not removing hydrogen atom without neighbors
[17:12:01] WARNING: not removing hydrogen atom without neighbors
[17:12:02] WARNING: not removing hydrogen atom without neighbors
[17:12:02] WARNING: not removing hydrogen atom without neighbors
[17:12:02] WARNING: not removing hydrogen atom without neighbors
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWa

...

[17:12:18] WARNING: not removing hydrogen atom without neighbors
[17:12:18] WARNING: not removing hydrogen atom without neighbors
[17:12:19] WARNING: not removing hydrogen atom without neighbors
[17:12:19] WARNING: not removing hydrogen atom without neighbors
[17:12:20] WARNING: not removing hydrogen atom without neighbors
[17:12:20] WARNING: not removing hydrogen atom without neighbors
[17:12:20] WARNING: not removing hydrogen atom without neighbors
[17:12:20] WARNING: not removing hydrogen atom without neighbors
[17:12:20] WARNING: not removing hydrogen atom without neighbors
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[17:12:21] WARNING: not removing hydrogen atom without neighbors
[17:12:21] WARNING: not removing hydrogen atom without neighbors
[17:12:21] WARNING: not removing hydrogen atom without neighbors
[17:12:21] WARNING: not removing hydrogen atom without neighbors
[17:12:21] WARNING: not removing hydrogen atom without neighbors
[17:12:21] WARNING: not removing hydrogen atom without neighbors
[17:12:21] WARNING: not removing hydrogen atom without neighbors
[17:12:21] WARNING: not removing hydrogen atom without neighbors
[17:12:21] WARNING: not removing hydrogen atom without neighbors
[17:12:22] WARNING: not removing hydrogen atom without neighbors
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=510, training_loss=0.601138479569379, metrics={'train_runtime': 26.3188, 'train_samples_per_second': 620.09, 'train_steps_per_second': 19.378, 'total_flos': 37597093724160.0, 'train_loss': 0.601138479569379, 'epoch': 10.0})

## 13. Evaluatie op testset

In [21]:
test_metrics = trainer.evaluate(test_ds)

print("=== Testresultaten ===")
for k, v in test_metrics.items():
    if isinstance(v, float):
        print(f"{k:25s}: {v:.4f}")

=== Testresultaten ===
eval_loss                : 0.5580
eval_accuracy            : 0.9261
eval_roc_auc             : nan
eval_mcc                 : 0.0000
eval_runtime             : 0.1131
eval_samples_per_second  : 1795.5940
eval_steps_per_second    : 35.3810
epoch                    : 10.0000


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(


In [22]:
from sklearn.metrics import confusion_matrix, classification_report

predictions = trainer.predict(test_ds)
y_pred = np.argmax(predictions.predictions, axis=-1)
y_true = predictions.label_ids

cm = confusion_matrix(y_true, y_pred)
print("Confusion matrix:")
print("                 voorspeld 0   voorspeld 1")
print(f"  werkelijk 0:    {cm[0,0]:5d}        {cm[0,1]:5d}")
print(f"  werkelijk 1:    {cm[1,0]:5d}        {cm[1,1]:5d}")

print("\nPer-klasse metrics:")
print(classification_report(y_true, y_pred, target_names=["geen BBB (0)", "wel BBB (1)"]))

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(


Confusion matrix:
                 voorspeld 0   voorspeld 1
  werkelijk 0:        0            0
  werkelijk 1:       15          188

Per-klasse metrics:
              precision    recall  f1-score   support

geen BBB (0)       0.00      0.00      0.00         0
 wel BBB (1)       1.00      0.93      0.96       203

    accuracy                           0.93       203
   macro avg       0.50      0.46      0.48       203
weighted avg       1.00      0.93      0.96       203



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
